1. Write Ansible playbooks to automate the setup and configuration of a web server (e.g., Apache or Nginx)?
```
Ansible playbook (Nginx web server):
---
- name: Setup and configure Nginx web server
  hosts: webservers
  become: yes

  vars:
    nginx_listen_port: 80
    nginx_server_name: "_"
    nginx_root: /var/www/html

  tasks:
    - name: Install Nginx
      apt:
        name: nginx
        state: present
        update_cache: yes
      when: ansible_os_family == "Debian"

    - name: Install Nginx (RHEL/CentOS)
      yum:
        name: nginx
        state: present
      when: ansible_os_family == "RedHat"

    - name: Ensure web root exists
      file:
        path: "{{ nginx_root }}"
        state: directory
        owner: www-data
        group: www-data
        mode: "0755"

    - name: Deploy index.html
      copy:
        dest: "{{ nginx_root }}/index.html"
        content: |
          <html>
          <head><title>Welcome</title></head>
          <body>
          <h1>Nginx via Ansible</h1>
          </body>
          </html>
        owner: www-data
        group: www-data
        mode: "0644"

    - name: Configure Nginx site
      copy:
        dest: /etc/nginx/sites-available/default
        content: |
          server {
              listen {{ nginx_listen_port }} default_server;
              listen [::]:{{ nginx_listen_port }} default_server;

              server_name {{ nginx_server_name }};
              root {{ nginx_root }};
              index index.html;

              location / {
                  try_files $uri $uri/ =404;
              }
          }
      notify: Reload nginx

    - name: Enable and start Nginx
      service:
        name: nginx
        state: started
        enabled: yes

  handlers:
    - name: Reload nginx
      service:
        name: nginx
        state: reloaded
```